# Rich

> Common `rich` patterns and helpers for MCP tool functions

In [ ]:
#| default_exp utils.rich

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple, Callable, TypeVar
from dataclasses import dataclass, field
from pathlib import Path
from functools import wraps
import io

from rich.table import Table
from rich.panel import Panel
from rich.console import Console
from rich.text import Text
from rich.markdown import Markdown


from nbdev_mcp.utils.types import ToolResult, Issue

## Result Types

Standardized result types for tool responses.

In [ ]:
#| export
def ok_result(**data) -> Dict[str, Any]:
    """Create a success dict with ok=True and additional data."""
    return {'ok': True, **data}

In [ ]:
#| export
def err_result(error: str, **data) -> Dict[str, Any]:
    """Create a failure dict with ok=False and error message."""
    return {'ok': False, 'error': error, **data}

## Project Resolution

Safe project resolution with standardized error handling.

In [ ]:
#| export
def try_resolve(selector: Optional[str], resolve_fn: Callable) -> Tuple[bool, Optional[Path], str]:
    """Safely resolve a project selector.
    
    Parameters
    ----------
    selector : str | None
        Project path, alias, or None for current project.
    resolve_fn : Callable
        Function to resolve selector to Path (usually resolve_selector).
    
    Returns
    -------
    Tuple[bool, Path | None, str]
        (success, path, error_message)
    """
    try:
        return True, resolve_fn(selector), ''
    except Exception as e:
        return False, None, str(e)

In [ ]:
#| export
T = TypeVar('T')

def with_project(resolve_fn: Callable):
    """Decorator that resolves project parameter and handles errors.
    
    The decorated function should have `project: Optional[str]` as first param.
    It will receive the resolved Path instead of the string.
    
    Example
    -------
    @with_project(resolve_selector)
    def my_tool(project: Path, other_arg: str) -> Dict[str, Any]:
        # project is now a resolved Path
        ...
    """
    def decorator(fn: Callable[..., T]) -> Callable[..., T]:
        @wraps(fn)
        def wrapper(project: Optional[str] = None, *args, **kwargs) -> T:
            ok, path, error = try_resolve(project, resolve_fn)
            if not ok:
                return {'ok': False, 'error': error}
            return fn(path, *args, **kwargs)
        return wrapper
    return decorator

## Rich Output Helpers

Helpers for generating pretty Rich console output.

In [ ]:
#| export
def make_console(width: int = 100) -> Console:
    """Create a Rich Console for capturing output (no stdout)."""
    return Console(file=io.StringIO(), force_terminal=True, width=width, record=True)

In [ ]:
#| export
def get_output(c: Console) -> str:
    """Extract recorded text from a Console."""
    return c.export_text(clear=False)

In [ ]:
#| export
def render_table(
    title: str,
    columns: List[str],
    rows: List[List[str]],
    max_rows: int = 200
) -> str:
    """Render a simple table as rich text.
    
    Parameters
    ----------
    title : str
        Table title.
    columns : List[str]
        Column headers.
    rows : List[List[str]]
        Row data (each row is a list of cell values).
    max_rows : int
        Maximum rows to display.
    
    Returns
    -------
    str
        Rendered table as text.
    """
    c = make_console()
    t = Table(title=title)
    for col in columns:
        t.add_column(col)
    for row in rows[:max_rows]:
        t.add_row(*[str(v) for v in row])
    c.print(t)
    return get_output(c)

In [ ]:
#| export
def render_dict_table(title: str, data: Dict[str, Any]) -> str:
    """Render a dict as a key-value table."""
    rows = [[k, str(v)] for k, v in data.items()]
    return render_table(title, ['Key', 'Value'], rows)

In [ ]:
#| export
def render_panel(title: str, content: str) -> str:
    """Render content in a titled panel."""
    c = make_console()
    c.print(Panel(content, title=title))
    return get_output(c)

## Lint Issue Types

Standardized types for lint/validation issues.

In [ ]:
#| export
def render_issues(issues: List[Issue], title: str = 'Issues') -> str:
    """Render a list of issues as a table."""
    rows = []
    for issue in issues:
        loc = issue.file or f"{issue.notebook}#cell{issue.cell}" if issue.notebook else ''
        rows.append([issue.rule, loc, issue.message])
    return render_table(title, ['Rule', 'Location', 'Message'], rows)

In [ ]:
#| export
def render_result(title: str, meta: Dict[str, Any], logs: Dict[str, Any] | None = None) -> str:
    """Render a result panel with metadata and optional command logs.
    
    Parameters
    ----------
    title : str
        Title for the result panel.
    meta : Dict[str, Any]
        Metadata to display in a key-value table.
    logs : Dict[str, Any] or None
        Command execution logs with keys: cmd, cwd, returncode, ok, stdout, stderr.
    
    Returns
    -------
    str
        Formatted Rich text output.
    """
    c = make_console()
    c.print(Panel.fit(Text(title, style="bold"), title="nbdev MCP"))
    if meta:
        t = Table(title="Context", expand=False)
        t.add_column("Key")
        t.add_column("Value")
        for k, v in meta.items():
            t.add_row(k, str(v))
        c.print(t)
    if logs:
        t2 = Table(title="Command", expand=False)
        t2.add_column("Field")
        t2.add_column("Value")
        for k in ("cmd", "cwd", "returncode", "ok"):
            if k in logs:
                t2.add_row(k, str(logs[k]))
        c.print(t2)
        if logs.get("stdout"):
            c.print(Panel.fit(Markdown(f"```\n{logs['stdout']}\n```"), title="stdout"))
        if logs.get("stderr"):
            c.print(Panel.fit(Markdown(f"```\n{logs['stderr']}\n```"), title="stderr"))
    return get_output(c)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()